# Notebook 4: `ai4bharat/indicconformer_stt_bn_hybrid_ctc_rnnt_large`

Family: `nemo_conformer` · **Python 3.11 required** · GPU recommended.

Settings → Environment → **Pin to original environment** before running.
The NeMo install cell is heavy (5-10 min) and the recipe is documented in DECISIONS.md §I. After the install, if Cell 4 prints `RESTART REQUIRED`, do that and re-run from the top.

Produces `predictions/ai4bharat__indicconformer_stt_bn_hybrid_ctc_rnnt_large.csv`.


In [ ]:
# === Python version check ===============================================
# This notebook needs Python 3.11. Kaggle's current default is 3.12,
# which has cp312 wheel gaps that make the NeMo install painful (see
# DECISIONS.md §I). Switch images: right sidebar → Settings →
# Environment → "Pin to original environment" (or pick a 3.11 image).

import sys
v = f"{sys.version_info.major}.{sys.version_info.minor}"
if v != "3.11":
    print("=" * 70)
    print(f"WARNING: detected Python {v}, this notebook expects 3.11.")
    print("Settings → Environment → Pin to original environment (3.11),")
    print("then Run → Restart & Clear, then re-run from the top.")
    print("=" * 70)
else:
    print(f"Python {v} OK.")


In [ ]:
# === Clone the benchmark repo ===========================================
# Kaggle paths: /kaggle/working is the writable home. We clone the project
# repo there and cd into it so the relative paths used by src/ resolve.

!apt-get -qq install -y libsndfile1

GITHUB_REPO_URL = "https://github.com/notAvailable73/stt-model-comparison.git"
REPO_DIR = "/kaggle/working/banglish-stt-benchmark"

import os, sys, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", GITHUB_REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd())


In [ ]:
# === Install AI4Bharat NeMo fork ========================================
# Heavy cell — 5-10 min if both workarounds apply, hours otherwise.
# Reason for the fork: model 4 needs the multi-softmax aggregate tokenizer
# + `language_id="bn"` kwarg that stock nemo_toolkit lacks. The fork
# installs editable under the same `nemo_toolkit` name, so hishab models
# 5/6 load on top — no second NeMo install. (DECISIONS.md §B)
#
# Two workarounds applied below (DECISIONS.md §I):
#
# (1) Pre-uninstall + legacy resolver. The fork pins huggingface_hub
#     ==0.23.2, but Kaggle ships transformers/datasets/diffusers built
#     against a much newer hf_hub. With pip's new resolver this triggers
#     multi-hour backtracking. Removing the conflict seeds first +
#     PIP_USE_DEPRECATED=legacy-resolver brings resolution back to seconds.
#
# (2) Relax tensorstore pin. The fork pins `tensorstore<0.1.46`, but
#     cp312 wheels only start at ~0.1.50 (cp311 wheels exist further back
#     but this guard catches both). Without it, pip falls back to a Bazel
#     source build that takes 30-60+ min.

!rm -rf /kaggle/working/NeMo
!git clone https://github.com/AI4Bharat/NeMo.git /kaggle/working/NeMo
!cd /kaggle/working/NeMo && git checkout nemo-v2

!pip uninstall -y transformers datasets diffusers huggingface_hub tokenizers
!pip install -q "pip<24.2"

!sed -i 's/tensorstore<0.1.46/tensorstore/g' /kaggle/working/NeMo/setup.py 2>/dev/null || true
!find /kaggle/working/NeMo/requirements -name '*.txt' -exec sed -i 's/tensorstore<0.1.46/tensorstore/g' {} +
!pip install -q "tensorstore>=0.1.71"

!cd /kaggle/working/NeMo && PIP_USE_DEPRECATED=legacy-resolver bash reinstall.sh

# Project's own pinned deps. reinstall.sh sometimes leaves numpy on an
# older ABI than the in-memory kernel — pin numpy here, and only touch
# torchvision if the in-memory torch <-> torchvision pair is actually broken.
!pip install -q transformers==4.44.2 accelerate==0.34.2 librosa==0.10.2.post1 \
    soundfile==0.12.1 pandas==2.2.3 pyyaml==6.0.2 tqdm==4.67.1 requests==2.32.3
!pip install -q --force-reinstall --no-deps numpy==2.0.2

# torchvision repair only if needed. reinstall.sh doesn't touch torch, so
# the resident torchvision is usually fine — but the previous version of
# this cell blindly --force-reinstalled the latest cu128 wheel, which is
# built against a newer torch than Kaggle ships and breaks torchvision::nms.
# Probe first; only repair if broken; then pin to the torchvision minor
# that matches the resident torch (table below covers torch 2.0 .. 2.11).
import torch, subprocess, sys
def _tv_ok():
    try:
        import torchvision  # noqa: F401
        return hasattr(torch.ops.torchvision, "nms")
    except Exception:
        return False

if _tv_ok():
    print(f"torchvision OK with torch={torch.__version__}")
else:
    _v = torch.__version__
    _base, _, _cu = _v.partition("+")
    _mm = ".".join(_base.split(".")[:2])
    _TV = {"2.0":"0.15","2.1":"0.16","2.2":"0.17","2.3":"0.18","2.4":"0.19",
           "2.5":"0.20","2.6":"0.21","2.7":"0.22","2.8":"0.23","2.9":"0.24",
           "2.10":"0.25","2.11":"0.26"}
    _tv = _TV.get(_mm)
    _cu = _cu or "cpu"
    _spec = f"torchvision=={_tv}.*" if _tv else "torchvision"
    print(f"torchvision broken (torch={_v}); installing {_spec} from cu={_cu}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "--force-reinstall", "--no-deps", _spec,
                           "--index-url", f"https://download.pytorch.org/whl/{_cu}"])


In [ ]:
# === Restart check (numpy ABI realignment) ==============================
# If the on-disk numpy differs from the in-memory numpy, restart the
# kernel and re-run all cells. Pip caches persist within a session, so
# the rerun is fast.

import importlib, subprocess, sys
_disk = subprocess.check_output(
    [sys.executable, "-c", "import numpy; print(numpy.__version__)"]
).decode().strip()
_runtime = importlib.import_module("numpy").__version__
print(f"numpy on disk: {_disk} | numpy in kernel: {_runtime}")
if _disk != _runtime:
    print("=" * 70)
    print("RESTART REQUIRED: Run → Restart, then re-run from the top.")
    print("=" * 70)
else:
    print("numpy aligned.")


In [ ]:
# === Make NeMo importable in this kernel ================================
# `pip install -e` writes a .pth file that's only read at interpreter
# startup. The kernel that ran the install above never sees it → import
# nemo fails. Add the source dir to sys.path manually for this session.
# (DECISIONS.md §I)

import sys
if "/kaggle/working/NeMo" not in sys.path:
    sys.path.append("/kaggle/working/NeMo")

# Verify
import nemo.collections.asr  # noqa: F401
print("nemo.collections.asr import OK")


In [ ]:
# === Hugging Face login (this model is gated) ===========================
# Steps before running:
#   1. Accept the license at
#      https://huggingface.co/ai4bharat/indicconformer_stt_bn_hybrid_ctc_rnnt_large
#   2. Create a read token at https://huggingface.co/settings/tokens
#   3. Kaggle sidebar -> Add-ons -> Secrets -> add HF_TOKEN, attach to notebook

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

login(token=UserSecretsClient().get_secret("HF_TOKEN"), add_to_git_credential=False)
print("HF login OK")


In [ ]:
# === Build the 50-clip manifest =========================================
# Downloads OpenSLR-104 if not already cached, extracts, parses transcripts,
# computes length + code-switch density, and stratified-samples 50 clips.
#
# CRITICAL: every notebook in this benchmark calls this with seed=42 so
# they all see the EXACT SAME 50 clips. Do not change the seed.

from src.utils import setup_logging, set_seeds
from src.data_prep import build_manifest

setup_logging("logs")
set_seeds(42)

manifest_path = build_manifest(
    raw_dir="data/raw",
    manifest_path="data/manifest.csv",
    n_samples=50,
    seed=42,
)
print("manifest:", manifest_path)


In [ ]:
# === Transcribe with this model =========================================
# Reads the spec for THIS notebook's model from config/models.yaml,
# then runs the 50-clip loop. Resumable: rerun this cell to pick up
# where it stopped (clips already in the CSV are skipped).

from src.utils import read_yaml
from src.transcribe import run_single_model

MODEL_ID = "ai4bharat/indicconformer_stt_bn_hybrid_ctc_rnnt_large"
spec = next(s for s in read_yaml("config/models.yaml")["models"] if s["id"] == MODEL_ID)

failure = run_single_model(
    spec,
    manifest_csv="data/manifest.csv",
    predictions_dir="predictions",
    device="cuda",
)
print("FAILED:", failure) if failure else print("OK — see predictions/")


In [ ]:
import os
from IPython.display import FileLink

# Define the exact path you provided
full_path = "/kaggle/working/banglish-stt-benchmark/predictions/ai4bharat__indicconformer_stt_bn_hybrid_ctc_rnnt_large.csv"
target_filename = "ai4bharat__indicconformer_stt_bn_hybrid_ctc_rnnt_large.csv"

if os.path.exists(full_path):
    # Copy the file to the root /kaggle/working/ directory so Kaggle can see it easily
    os.system(f'cp "{full_path}" "./{target_filename}"')
    print("File successfully copied to root directory!")
    
    # Generate the direct download link
    display(FileLink(target_filename))
else:
    print(f"Error: Could not find the file at {full_path}")
    print("Please double-check that the cell in image_657b83.png finished running completely.")
